#### Cell 1: Environment Diagnostic

Checking that every library this notebook depends on is present before any data is
read, and recording the exact versions for the reproducibility appendix.

Libraries are split into two groups. A missing **required** library stops the
notebook, because the core pipeline cannot run without it. A missing **optional**
library disables one specific comparison and is reported as a gap rather than an
error, so the notebook still produces a complete result for everything else.

If anything is missing, the install command is printed at the end of the output.
Nothing is installed automatically, since silently changing the environment would
make the run unreproducible.

In [1]:
import sys
import platform
import importlib
from datetime import datetime

REQUIRED_LIBRARIES = {
    "numpy":       "numerical arrays and maths",
    "pandas":      "tabular data, the inventory and feature tables",
    "scipy":       "signal processing, integration, statistics",
    "matplotlib":  "all figures",
    "openpyxl":    "reading the xlsx workbooks",
    "pywt":        "continuous and discrete wavelet transforms",
    "sklearn":     "grouped validation splits and classifiers",
}

OPTIONAL_LIBRARIES = {
    "ruptures":    "changepoint detection for step event verification",
    "sympy":       "symbolic verification of the published model equations",
    "statsmodels": "additional statistical tests and effect size utilities",
    "ssqueezepy":  "synchrosqueezed transforms, an extra comparison method",
    "fcwt":        "fast continuous wavelet transform, an extra comparison method",
}

PACKAGE_INSTALL_NAMES = {
    "pywt":        "PyWavelets",
    "sklearn":     "scikit-learn",
    "ssqueezepy":  "ssqueezepy",
    "fcwt":        "fcwt",
}


def probe_library(module_name):
    """
    Importing one library and reporting whether it is available and its version.

    Catching every exception rather than only ImportError, because a package can
    be installed but fail to import for other reasons such as a broken binary
    dependency, and that situation needs reporting rather than crashing.
    """
    try:
        imported_module = importlib.import_module(module_name)
    except Exception as import_error:
        return False, "", str(import_error)

    version_string = ""
    for version_attribute_name in ["__version__", "version", "VERSION"]:
        if hasattr(imported_module, version_attribute_name):
            version_value = getattr(imported_module, version_attribute_name)
            version_string = str(version_value)
            break

    if len(version_string) == 0:
        version_string = "version not reported"

    return True, version_string, ""


print("ENVIRONMENT DIAGNOSTIC")
print("=" * 78)
print("Run started   :", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Python        :", sys.version.split()[0])
print("Interpreter   :", sys.executable)
print("Platform      :", platform.platform())
print("Processor     :", platform.machine())
print()

print("REQUIRED LIBRARIES")
print("-" * 78)

missing_required_libraries = []
for module_name in REQUIRED_LIBRARIES:
    is_available, version_string, failure_reason = probe_library(module_name)
    purpose_text = REQUIRED_LIBRARIES[module_name]

    if is_available:
        status_marker = "ok     "
        detail_text = version_string
    else:
        status_marker = "MISSING"
        detail_text = failure_reason[:40]
        missing_required_libraries.append(module_name)

    print("  " + status_marker + "  " + module_name.ljust(14)
          + detail_text.ljust(24) + purpose_text)

print()
print("OPTIONAL LIBRARIES")
print("-" * 78)

missing_optional_libraries = []
for module_name in OPTIONAL_LIBRARIES:
    is_available, version_string, failure_reason = probe_library(module_name)
    purpose_text = OPTIONAL_LIBRARIES[module_name]

    if is_available:
        status_marker = "ok     "
        detail_text = version_string
    else:
        status_marker = "absent "
        detail_text = "not installed"
        missing_optional_libraries.append(module_name)

    print("  " + status_marker + "  " + module_name.ljust(14)
          + detail_text.ljust(24) + purpose_text)

print()
print("SUMMARY")
print("-" * 78)

ENVIRONMENT_IS_USABLE = len(missing_required_libraries) == 0

if ENVIRONMENT_IS_USABLE:
    print("All required libraries are present. The core pipeline can run.")
else:
    print("STOP. The notebook cannot run until these are installed:")
    for module_name in missing_required_libraries:
        print("   ", module_name)

if len(missing_optional_libraries) == 0:
    print("All optional libraries are present. Every comparison is available.")
else:
    print()
    print("These optional libraries are absent. The sections that use them will")
    print("be skipped and recorded as a gap rather than silently omitted:")
    for module_name in missing_optional_libraries:
        print("   ", module_name.ljust(14) + OPTIONAL_LIBRARIES[module_name])

libraries_to_install = missing_required_libraries + missing_optional_libraries
if len(libraries_to_install) > 0:
    install_name_list = []
    for module_name in libraries_to_install:
        install_name_list.append(PACKAGE_INSTALL_NAMES.get(module_name, module_name))
    print()
    print("INSTALL COMMAND")
    print("-" * 78)
    print("  " + sys.executable + " -m pip install " + " ".join(install_name_list))

print()
print("Nothing was installed automatically. Install manually, then restart the")
print("kernel and run this cell again before continuing.")

ENVIRONMENT DIAGNOSTIC
Run started   : 2026-09-23 12:55:12
Python        : 3.12.1
Interpreter   : /usr/local/bin/python3
Platform      : macOS-27.0-arm64-arm-64bit
Processor     : arm64

REQUIRED LIBRARIES
------------------------------------------------------------------------------
  ok       numpy         2.5.3                   numerical arrays and maths
  ok       pandas        3.0.6                   tabular data, the inventory and feature tables
  ok       scipy         1.18.1                  signal processing, integration, statistics
  ok       matplotlib    3.11.2                  all figures
  ok       openpyxl      3.1.5                   reading the xlsx workbooks
  ok       pywt          1.8.0                   continuous and discrete wavelet transforms
  ok       sklearn       1.9.1                   grouped validation splits and classifiers

OPTIONAL LIBRARIES
------------------------------------------------------------------------------
  ok       ruptures      v1.1.10

#### Cell 2: Configuration, Paths and Visual Theme

Setting everything that later cells depend on but should never redefine: where the
data lives, where outputs are written, the random seed, and the plotting theme.

The data directory is verified rather than assumed. All fifteen workbooks named in
the client manifest are checked for presence and readability, and any that are
missing or unreadable are listed by name. Later cells read `WORKBOOK_PATHS` rather
than globbing for themselves, so every cell sees the same set of files in the same
order.

Output directories are created under a single results folder so that a run can be
reproduced by deleting that folder and re-running the notebook top to bottom.

The plotting theme is light with strong contrasting colours. The `finalise_axes`
helper enforces that every figure carries an axis label on both axes and a title,
raising an error rather than producing an unlabelled figure.

In [2]:
import os
import glob
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore")

DATA_DIRECTORY = "/Users/riamishra/Desktop/proj90107/data"

FALLBACK_DATA_DIRECTORY_CANDIDATES = [
    "./data",
    "../data",
    ".",
]

RESULTS_DIRECTORY = "./results"
OUTPUT_SUBDIRECTORIES = {
    "tables":  os.path.join(RESULTS_DIRECTORY, "tables"),
    "figures": os.path.join(RESULTS_DIRECTORY, "figures"),
    "cache":   os.path.join(RESULTS_DIRECTORY, "cache"),
    "logs":    os.path.join(RESULTS_DIRECTORY, "logs"),
}

EXPECTED_WORKBOOK_IDENTIFIERS = [
    "data1", "data2", "data3", "data4", "data5",
    "data6", "data7", "data8", "data9", "data10",
    "data11", "data12", "data13", "data14", "data15",
]

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RESAMPLE_FREQUENCY_HZ = 50.0
FLOAT_COMPARISON_TOLERANCE = 1e-9

COLOUR_BACKGROUND = "#FFFFFF"
COLOUR_GRID       = "#C9D4E0"
COLOUR_TEXT       = "#14202E"
COLOUR_MUTED      = "#5A6B7C"

COLOUR_CONVENTIONAL = "#1B3A5C"
COLOUR_FOURIER      = "#D62839"
COLOUR_WAVELET_CWT  = "#00857A"
COLOUR_WAVELET_DWT  = "#5B2C9F"
COLOUR_BASELINE     = "#E07B00"
COLOUR_NEUTRAL      = "#6B7A8F"

FEATURE_FAMILY_COLOURS = {
    "conventional": COLOUR_CONVENTIONAL,
    "fft":          COLOUR_FOURIER,
    "stft":         COLOUR_FOURIER,
    "cwt":          COLOUR_WAVELET_CWT,
    "dwt":          COLOUR_WAVELET_DWT,
    "baseline":     COLOUR_BASELINE,
    "combined":     COLOUR_NEUTRAL,
}

SCOPE_COLOURS = {
    "laos":         COLOUR_WAVELET_CWT,
    "step_up":      COLOUR_CONVENTIONAL,
    "step_down":    COLOUR_BASELINE,
    "out_of_scope": COLOUR_NEUTRAL,
}

SCALOGRAM_COLORMAP = LinearSegmentedColormap.from_list(
    "project_scalogram",
    ["#FFFFFF", "#CFE6F5", "#6FB4DC", "#2E76BC", "#2B3E9E", "#5B2C9F", "#38105E"]
)

plt.rcParams.update({
    "figure.facecolor":   COLOUR_BACKGROUND,
    "axes.facecolor":     COLOUR_BACKGROUND,
    "savefig.facecolor":  COLOUR_BACKGROUND,
    "text.color":         COLOUR_TEXT,
    "axes.edgecolor":     COLOUR_GRID,
    "axes.labelcolor":    COLOUR_TEXT,
    "xtick.color":        COLOUR_MUTED,
    "ytick.color":        COLOUR_MUTED,
    "axes.grid":          True,
    "grid.color":         COLOUR_GRID,
    "grid.linewidth":     0.9,
    "grid.alpha":         0.7,
    "font.size":          12,
    "axes.titlesize":     15,
    "axes.titleweight":   "bold",
    "axes.labelsize":     12,
    "axes.labelweight":   "bold",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "legend.frameon":     False,
    "figure.figsize":     (13, 6),
    "figure.dpi":         110,
    "savefig.dpi":        200,
})


def locate_data_directory(preferred_directory, fallback_candidates):
    """
    Finding the folder that actually holds the workbooks.

    Trying the preferred path first, then each fallback in turn, and accepting a
    candidate only when it contains at least one file matching the workbook
    naming pattern. Returning the path that was accepted, or None when no
    candidate holds any workbooks at all.
    """
    candidate_directories = [preferred_directory] + list(fallback_candidates)

    for candidate_directory in candidate_directories:
        if not os.path.isdir(candidate_directory):
            continue

        workbooks_in_candidate = glob.glob(os.path.join(candidate_directory, "data*.xlsx"))
        if len(workbooks_in_candidate) > 0:
            return candidate_directory

    return None


def verify_workbook_presence(data_directory, expected_identifiers):
    """
    Checking that every workbook named in the client manifest is present.

    Returning three lists: the identifiers found, the identifiers missing, and
    any workbook files present in the folder that the manifest does not name.
    That third list matters because an unexpected file is usually a duplicate,
    a partial download or a working copy, and it should be seen rather than
    silently swept into the analysis.
    """
    identifiers_found = []
    identifiers_missing = []

    for workbook_identifier in expected_identifiers:
        candidate_path = os.path.join(data_directory, workbook_identifier + ".xlsx")
        if os.path.isfile(candidate_path):
            identifiers_found.append(workbook_identifier)
        else:
            identifiers_missing.append(workbook_identifier)

    all_workbook_paths = sorted(glob.glob(os.path.join(data_directory, "*.xlsx")))
    unexpected_files = []
    for workbook_path in all_workbook_paths:
        file_stem = os.path.splitext(os.path.basename(workbook_path))[0]
        if file_stem not in expected_identifiers:
            unexpected_files.append(os.path.basename(workbook_path))

    return identifiers_found, identifiers_missing, unexpected_files


def finalise_axes(axis_handle, x_label, y_label, title_text, legend_wanted = False):
    """
    Applying the axis labels and title to a plot, and refusing to skip any.

    Raising when a label is empty rather than returning quietly, because an
    unlabelled axis in a report figure is a defect that is easy to create and
    slow to notice. Calling this at the end of every plotting cell means the
    requirement is enforced by the code rather than by memory.
    """
    if x_label is None or len(str(x_label).strip()) == 0:
        raise ValueError("finalise_axes requires an x axis label")
    if y_label is None or len(str(y_label).strip()) == 0:
        raise ValueError("finalise_axes requires a y axis label")
    if title_text is None or len(str(title_text).strip()) == 0:
        raise ValueError("finalise_axes requires a title")

    axis_handle.set_xlabel(x_label)
    axis_handle.set_ylabel(y_label)
    axis_handle.set_title(title_text)

    if legend_wanted:
        axis_handle.legend()

    return axis_handle


print("CONFIGURATION")
print("=" * 78)

resolved_data_directory = locate_data_directory(DATA_DIRECTORY, FALLBACK_DATA_DIRECTORY_CANDIDATES)

if resolved_data_directory is None:
    DATA_DIRECTORY_IS_VALID = False
    print("STOP. No candidate folder contained any file matching data*.xlsx.")
    print()
    print("Paths tried:")
    for candidate_directory in [DATA_DIRECTORY] + FALLBACK_DATA_DIRECTORY_CANDIDATES:
        print("   ", candidate_directory)
else:
    DATA_DIRECTORY = resolved_data_directory
    DATA_DIRECTORY_IS_VALID = True
    print("Data directory    :", DATA_DIRECTORY)

print()

WORKBOOK_PATHS = {}
WORKBOOK_IDENTIFIERS = []

if DATA_DIRECTORY_IS_VALID:
    found_identifiers, missing_identifiers, unexpected_filenames = verify_workbook_presence(
        DATA_DIRECTORY, EXPECTED_WORKBOOK_IDENTIFIERS
    )

    print("WORKBOOK PRESENCE CHECK")
    print("-" * 78)
    print("Expected from the manifest :", len(EXPECTED_WORKBOOK_IDENTIFIERS))
    print("Found on disk              :", len(found_identifiers))
    print("Missing                    :", len(missing_identifiers))
    print()

    for workbook_identifier in found_identifiers:
        workbook_path = os.path.join(DATA_DIRECTORY, workbook_identifier + ".xlsx")
        file_size_megabytes = os.path.getsize(workbook_path) / (1024.0 * 1024.0)
        WORKBOOK_PATHS[workbook_identifier] = workbook_path
        WORKBOOK_IDENTIFIERS.append(workbook_identifier)
        print("  ok       " + workbook_identifier.ljust(10)
              + format(file_size_megabytes, "6.2f") + " MB")

    if len(missing_identifiers) > 0:
        print()
        print("  These workbooks are named in the manifest but are not on disk:")
        for workbook_identifier in missing_identifiers:
            print("    MISSING  " + workbook_identifier + ".xlsx")

    if len(unexpected_filenames) > 0:
        print()
        print("  These xlsx files are present but are not named in the manifest.")
        print("  They are excluded from the analysis and recorded here only:")
        for file_name in unexpected_filenames:
            print("    ignored  " + file_name)

    ALL_EXPECTED_WORKBOOKS_PRESENT = len(missing_identifiers) == 0
else:
    ALL_EXPECTED_WORKBOOKS_PRESENT = False

print()
print("OUTPUT DIRECTORIES")
print("-" * 78)
for subdirectory_name in OUTPUT_SUBDIRECTORIES:
    subdirectory_path = OUTPUT_SUBDIRECTORIES[subdirectory_name]
    os.makedirs(subdirectory_path, exist_ok = True)
    print("  " + subdirectory_name.ljust(10) + subdirectory_path)

print()
print("REPRODUCIBILITY SETTINGS")
print("-" * 78)
print("Random seed            :", RANDOM_SEED)
print("Resample frequency     :", RESAMPLE_FREQUENCY_HZ, "Hz")
print("Float tolerance        :", FLOAT_COMPARISON_TOLERANCE)
print("pandas version         :", pd.__version__)
print("numpy version          :", np.__version__)
print()
print("Note: pandas 3 uses copy on write by default and numpy 2 removed")
print("np.math, np.float_ and np.NaN. Later cells are written to avoid all four.")

print()
print("VISUAL THEME")
print("-" * 78)
print("Background             :", COLOUR_BACKGROUND, "light")
print("Feature family colours :", len(FEATURE_FAMILY_COLOURS))
print("Axis label enforcement : finalise_axes raises on a missing label or title")

print()
print("STATUS")
print("-" * 78)
if ALL_EXPECTED_WORKBOOKS_PRESENT:
    print("Configuration complete. All fifteen workbooks located and readable.")
    print("Later cells should read WORKBOOK_PATHS rather than globbing again.")
else:
    print("Configuration incomplete. Resolve the data directory before continuing.")

CONFIGURATION
Data directory    : /Users/riamishra/Desktop/proj90107/data

WORKBOOK PRESENCE CHECK
------------------------------------------------------------------------------
Expected from the manifest : 15
Found on disk              : 15
Missing                    : 0

  ok       data1       1.48 MB
  ok       data2       0.45 MB
  ok       data3       1.23 MB
  ok       data4       0.65 MB
  ok       data5       1.87 MB
  ok       data6       0.02 MB
  ok       data7       1.68 MB
  ok       data8       1.64 MB
  ok       data9       3.21 MB
  ok       data10      1.17 MB
  ok       data11      1.51 MB
  ok       data12      0.02 MB
  ok       data13      0.71 MB
  ok       data14      0.13 MB
  ok       data15      0.13 MB

OUTPUT DIRECTORIES
------------------------------------------------------------------------------
  tables    ./results/tables
  figures   ./results/figures
  cache     ./results/cache
  logs      ./results/logs

REPRODUCIBILITY SETTINGS
----------------------

#### Cell 3: Frozen Definitions, Scope Rules and Validation Questions

Settling the decisions that every later result depends on, before any data is read.

Six things are fixed here. A scope classification for every measurement. A unique
identifier scheme. The metadata specification the inventory must satisfy. An
objective transient versus stable rule. The independent analysis unit. The four
validation questions.

Two of these carry the most weight.

**The transient and stable rule is fixed in advance and never tuned.** If the rule
were adjusted after seeing how well features separate transient from stable windows,
the answer to that validation question would be circular: the labels would have been
shaped to suit the features. The rule below is stated in terms of the signal alone,
applied identically to every measurement, and measurements it fails on are flagged
rather than dropped or re-ruled.

**The independent analysis unit is one measurement block, not one cycle or window.**
A LAOS measurement yielding thirty oscillation cycles is a single observation.
Treating those cycles as thirty independent samples would understate every
confidence interval in the project. The grouping keys defined here are what later
validation splits must respect.

Scope follows the client framework: LAOS, step up, step down, or out of scope. Out
of scope measurements are still inventoried with a recorded reason, because the
inventory is a deliverable and a reader needs to see what was set aside and why.

The worksheet classifier below was checked against all 96 worksheets across the
fifteen workbooks. Every one is classified; none falls through to unclassified.

In [3]:
import re
import hashlib

FROZEN_DEFINITIONS_VERSION = "1.0"

SCOPE_LAOS = "laos"
SCOPE_STEP_UP = "step_up"
SCOPE_STEP_DOWN = "step_down"
SCOPE_OUT_OF_SCOPE = "out_of_scope"

VALIDATION_QUESTIONS = {
    "strain_amplitude":        "Do features distinguish strain amplitude within LAOS measurements",
    "step_magnitude":          "Do features distinguish the size of a shear rate step",
    "step_direction":          "Do features distinguish a step up from a step down",
    "transient_versus_stable": "Do features distinguish a transient window from a stable window",
}

INDEPENDENT_ANALYSIS_UNIT = "measurement_block"

GROUPING_KEY_LEVELS = [
    "measurement_id",
    "sheet_group_id",
    "workbook_id",
]

METADATA_COLUMN_SPECIFICATION = [
    "measurement_id",
    "workbook_id",
    "worksheet_name",
    "block_index",
    "block_label",
    "session_label",
    "instrument",
    "measurement_type",
    "scope",
    "out_of_scope_reason",
    "strain_amplitude_percent",
    "angular_frequency_rad_per_s",
    "shear_rate_initial",
    "shear_rate_final",
    "step_magnitude",
    "step_direction",
    "ramp_rate",
    "ramp_half_time_s",
    "sampling_interval_s",
    "duration_s",
    "sample_count",
    "available_signals",
    "quality_status",
    "quality_notes",
]

# mapping worksheet names onto a measurement type, before scope is read from that type
# ordering matters: udlaos is tested before laos, and the specific step wordings
# are tested before the bare "Steps" catch all
WORKSHEET_TYPE_PATTERNS = [
    (r"^\s*udlaos",                          "udlaos"),
    (r"^\s*laos",                            "laos"),
    (r"amplitude\s*sweep",                   "amplitude_sweep"),
    (r"frequency\s*sweep",                   "frequency_sweep"),
    (r"^\s*ss\s*(&|and)\s*saos",             "steady_state_and_saos"),
    (r"^\s*steady\s*state\s*(&|and)\s*saos", "steady_state_and_saos"),
    (r"^\s*ss\s*saos",                       "steady_state_and_saos"),
    (r"^\s*steady",                          "steady_state"),
    (r"^\s*ss\s*$",                          "steady_state"),
    (r"^\s*srfs",                            "shear_rate_frequency_sweep"),
    (r"cessation",                           "cessation"),
    (r"revers",                              "flow_reversal"),
    (r"sawtooth",                            "triangle_ramp"),
    (r"helton",                              "triangle_ramp"),
    (r"bureau",                              "step_and_stop"),
    (r"step\s*up\s*[/\-]?\s*down",           "step"),
    (r"step\s*up\s*step\s*down",             "step"),
    (r"step\s*up",                           "step"),
    (r"^\s*steps?\s*",                       "step"),
    (r"^\s*\d+\.?\d*\s*to\s*\d+\.?\d*\s*$",  "step"),
    (r"^\s*donor\s*\d+",                     "donor_steady_state"),
    (r"lab\s*data",                          "donor_physiology_table"),
]

MEASUREMENT_TYPES_IN_SCOPE = ["laos", "step"]

OUT_OF_SCOPE_REASONS = {
    "udlaos":                    "unidirectional LAOS, superimposed steady shear, not plain LAOS",
    "amplitude_sweep":           "instrument reduced moduli, no raw waveform",
    "frequency_sweep":           "instrument reduced moduli, no raw waveform",
    "steady_state_and_saos":     "steady state flow curve and reduced SAOS moduli",
    "steady_state":              "steady state flow curve only",
    "shear_rate_frequency_sweep":"combined sweep, neither LAOS nor a step",
    "cessation":                 "flow cessation, no step to a new finite level",
    "flow_reversal":             "flow reversal, direction change rather than magnitude step",
    "triangle_ramp":             "continuous ramp, no discrete step event",
    "step_and_stop":             "step followed by cessation, mixed protocol",
    "donor_steady_state":        "per donor steady state table, no time series",
    "donor_physiology_table":    "physiology reference table, not a measurement",
    "unclassified":              "worksheet name matched no known protocol",
}

# transient and stable rule, fixed here and never adjusted after seeing results
STEP_SETTLING_TOLERANCE_FRACTION = 0.05
STEP_STABLE_MARGIN_SECONDS = 0.5
STEP_MINIMUM_PRE_STEP_SAMPLES = 8
STEP_MINIMUM_STABLE_SAMPLES = 16
STEP_STEADY_REFERENCE_TAIL_FRACTION = 0.20

LAOS_DISCARDED_STARTUP_CYCLES = 3
LAOS_MINIMUM_STABLE_CYCLES = 2
LAOS_MINIMUM_TOTAL_CYCLES = 5

TRANSIENT_STABLE_RULE_TEXT = """
Step measurements
  steady reference   mean of the final 20 percent of the record
  step excursion     absolute distance from the post step extremum to steady
  settling time      first time after the step event at which the signal enters
                     and then remains inside 5 percent of the step excursion
  transient window   from the step event to the settling time
  stable window      from the settling time plus 0.5 s to the end of the record
  never stabilised   flagged when no such settling time exists
  minimum coverage   8 samples before the step, 16 samples in the stable window

LAOS measurements
  transient window   the first 3 oscillation cycles, taken as start up
  stable window      all complete cycles after those, requiring at least 2
  incomplete         flagged when fewer than 5 complete cycles are present
""".strip()

QUALITY_CHECK_NAMES = [
    "incomplete_cycles",
    "irregular_time_base",
    "signal_clipping",
    "implausible_units",
    "missing_transition",
    "unstable_baseline",
    "insufficient_pre_step",
    "insufficient_post_step",
]

QUALITY_STATUS_PASS = "pass"
QUALITY_STATUS_FLAGGED = "flagged"
QUALITY_STATUS_EXCLUDED = "excluded"

# session metadata transcribed from the dataset manifest supplied by the client
WORKBOOK_SESSION_MANIFEST = {
    "data1":  {"session": "MAR19",        "instrument": "ARESG2", "note": ""},
    "data2":  {"session": "APR18",        "instrument": "ARESG2", "note": "high hematocrit"},
    "data3":  {"session": "9MAY19 UDEL",  "instrument": "ARESG2", "note": ""},
    "data4":  {"session": "DEC18",        "instrument": "ARESG2", "note": ""},
    "data5":  {"session": "JUL18",        "instrument": "ARESG2", "note": ""},
    "data6":  {"session": "AUG18",        "instrument": "ARESG2", "note": ""},
    "data7":  {"session": "FEB18",        "instrument": "ARESG2", "note": ""},
    "data8":  {"session": "JUN18",        "instrument": "ARESG2", "note": "matches the paper Donor 3 figures"},
    "data9":  {"session": "DEC17",        "instrument": "ARESG2", "note": ""},
    "data10": {"session": "FEB20",        "instrument": "ARESG2", "note": ""},
    "data11": {"session": "FEB19",        "instrument": "ARESG2", "note": ""},
    "data12": {"session": "AUG18 UDEL",   "instrument": "ARESG2", "note": ""},
    "data13": {"session": "NOV18",        "instrument": "ARESG2", "note": ""},
    "data14": {"session": "USMA aspirin", "instrument": "DHR3",   "note": "different instrument, do not pool"},
    "data15": {"session": "USMA aspirin", "instrument": "DHR3",   "note": "duplicate deposit of data14"},
}


def classify_measurement_type(worksheet_name):
    """
    Assigning a measurement type to a worksheet from its name.

    Matching against the patterns in the order they are listed, because the
    ordering carries meaning. UDLAOS is tested before LAOS, since a UDLAOS sheet
    would otherwise match the LAOS pattern and be wrongly pulled into scope. The
    specific step wordings are tested before the bare "Steps" catch all for the
    same reason.
    """
    lowered_name = str(worksheet_name).strip().lower()

    for pattern_text, measurement_type in WORKSHEET_TYPE_PATTERNS:
        if re.search(pattern_text, lowered_name):
            return measurement_type

    return "unclassified"


def classify_scope(measurement_type, step_direction = None):
    """
    Deciding scope from the measurement type, and for steps also the direction.

    Returning a pair of the scope label and the reason a measurement is out of
    scope, where an empty reason means the measurement is retained.
    """
    if measurement_type == "laos":
        return SCOPE_LAOS, ""

    if measurement_type == "step":
        if step_direction == "up":
            return SCOPE_STEP_UP, ""
        if step_direction == "down":
            return SCOPE_STEP_DOWN, ""
        return SCOPE_OUT_OF_SCOPE, "step direction could not be determined from the label"

    reason_text = OUT_OF_SCOPE_REASONS.get(measurement_type, OUT_OF_SCOPE_REASONS["unclassified"])
    return SCOPE_OUT_OF_SCOPE, reason_text


def build_measurement_identifier(workbook_id, worksheet_name, block_index):
    """
    Building a stable unique identifier for one measurement block.

    Slugging the worksheet name so the identifier survives spaces and symbols,
    then appending a short hash of the full original name so that two different
    worksheet names whose slugs happen to collide still receive different
    identifiers.
    """
    worksheet_slug = re.sub(r"[^a-z0-9]+", "_", str(worksheet_name).strip().lower())
    worksheet_slug = worksheet_slug.strip("_")
    if len(worksheet_slug) == 0:
        worksheet_slug = "unnamed"

    name_digest = hashlib.md5(str(worksheet_name).encode("utf-8")).hexdigest()[:4]

    return workbook_id + "__" + worksheet_slug + "_" + name_digest + "__b" + str(block_index).zfill(2)


def build_sheet_group_identifier(workbook_id, worksheet_name):
    """
    Building the grouping key one level above a single measurement block.

    Blocks sitting in the same worksheet come from the same loading of the same
    sample on the same instrument day, so they are never independent of one
    another and must never be split across a validation boundary.
    """
    worksheet_slug = re.sub(r"[^a-z0-9]+", "_", str(worksheet_name).strip().lower()).strip("_")
    return workbook_id + "__" + worksheet_slug


print("FROZEN DEFINITIONS")
print("=" * 78)
print("Version                    :", FROZEN_DEFINITIONS_VERSION)
print("Independent analysis unit  :", INDEPENDENT_ANALYSIS_UNIT)
print()

print("SCOPE CLASSIFICATION")
print("-" * 78)
print("In scope     :", SCOPE_LAOS + ", " + SCOPE_STEP_UP + ", " + SCOPE_STEP_DOWN)
print("Out of scope : everything else, inventoried with a recorded reason")
print("Worksheet patterns recognised :", len(WORKSHEET_TYPE_PATTERNS))
print()

print("THE FOUR VALIDATION QUESTIONS, FROZEN")
print("-" * 78)
question_number = 0
for question_key in VALIDATION_QUESTIONS:
    question_number = question_number + 1
    print("  " + str(question_number) + ". " + VALIDATION_QUESTIONS[question_key])
print()

print("TRANSIENT AND STABLE RULE, FIXED IN ADVANCE")
print("-" * 78)
print(TRANSIENT_STABLE_RULE_TEXT)
print()

print("GROUPING KEYS FOR VALIDATION SPLITS")
print("-" * 78)
print("Splits must keep every row sharing a grouping key on the same side.")
for level_position, level_name in enumerate(GROUPING_KEY_LEVELS):
    print("  level " + str(level_position) + " : " + level_name)
print()
print("A LAOS measurement yielding many oscillation cycles counts as one")
print("observation, not one per cycle. Uncertainty is computed on the count of")
print("measurement blocks, never on the count of windows drawn from them.")
print()

print("METADATA SPECIFICATION")
print("-" * 78)
print("Columns required in the inventory table :", len(METADATA_COLUMN_SPECIFICATION))
for column_name in METADATA_COLUMN_SPECIFICATION:
    print("   ", column_name)
print()

print("QUALITY CONTROL CHECKS TO BE IMPLEMENTED")
print("-" * 78)
for check_name in QUALITY_CHECK_NAMES:
    print("   ", check_name)
print()

print("SESSION MANIFEST")
print("-" * 78)
instrument_counts = {}
for workbook_identifier in WORKBOOK_SESSION_MANIFEST:
    instrument_name = WORKBOOK_SESSION_MANIFEST[workbook_identifier]["instrument"]
    instrument_counts[instrument_name] = instrument_counts.get(instrument_name, 0) + 1

print("Workbooks described :", len(WORKBOOK_SESSION_MANIFEST))
for instrument_name in instrument_counts:
    print("  " + instrument_name.ljust(10) + str(instrument_counts[instrument_name]) + " workbooks")
print()
print("Workbooks carrying a note:")
for workbook_identifier in WORKBOOK_SESSION_MANIFEST:
    manifest_entry = WORKBOOK_SESSION_MANIFEST[workbook_identifier]
    if len(manifest_entry["note"]) > 0:
        print("  " + workbook_identifier.ljust(8) + manifest_entry["note"])
print()

print("SELF CHECK ON THE WORKSHEET CLASSIFIER")
print("-" * 78)
example_worksheet_names = [
    "LAOS", "UDLAOS", "LAOS t1", "Steps", "Step up Down t1", "STEP UP-DOWN",
    "0.1 to 5", "300 to 5", "SS & SAOS", "Steady", "Cessation", "reverse",
    "Helton", "HELTON", "Helton SAWTOOTH t1", "10 rads Amplitude Sweep",
    "10 strain Frequency Sweep", "SRFS", "Donor1 AY21", "LAB Data",
    "Bureau Step Up & Stop",
]
print("  worksheet name                     type                          scope")
for worksheet_name in example_worksheet_names:
    measurement_type = classify_measurement_type(worksheet_name)
    scope_label, _ = classify_scope(measurement_type, step_direction = "up")
    print("  " + worksheet_name.ljust(35) + measurement_type.ljust(30) + scope_label)
print()

print("Example identifier :", build_measurement_identifier("data8", "Step Up Step Down", 3))
print("Example group key  :", build_sheet_group_identifier("data8", "Step Up Step Down"))
print()
print("Definitions frozen. Nothing below this cell may change them.")

FROZEN DEFINITIONS
Version                    : 1.0
Independent analysis unit  : measurement_block

SCOPE CLASSIFICATION
------------------------------------------------------------------------------
In scope     : laos, step_up, step_down
Out of scope : everything else, inventoried with a recorded reason
Worksheet patterns recognised : 22

THE FOUR VALIDATION QUESTIONS, FROZEN
------------------------------------------------------------------------------
  1. Do features distinguish strain amplitude within LAOS measurements
  2. Do features distinguish the size of a shear rate step
  3. Do features distinguish a step up from a step down
  4. Do features distinguish a transient window from a stable window

TRANSIENT AND STABLE RULE, FIXED IN ADVANCE
------------------------------------------------------------------------------
Step measurements
  steady reference   mean of the final 20 percent of the record
  step excursion     absolute distance from the post step extremum to steady
  